# India Runs — Build Dense BGE-large Index on T4 GPU

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Upload `data/processed/challenge_candidates.parquet` from your local machine when Cell 2 prompts you

**Output:** Two files you download and copy to `outputs/models/` on your laptop:
- `dense_demo_index.faiss`
- `dense_demo_index.meta.pkl`

**Expected time:** ~5 minutes on T4 GPU

In [ ]:
# Cell 1 — Install dependencies
!pip install -q sentence-transformers faiss-gpu pyarrow pandas

In [ ]:
# Cell 2 — Upload the candidate parquet
# Upload: data/processed/challenge_candidates.parquet from your local machine
from google.colab import files
uploaded = files.upload()   # select challenge_candidates.parquet
print('Uploaded:', list(uploaded.keys()))

In [ ]:
# Cell 3 — Load candidates and build text for each profile
import pandas as pd
import numpy as np

df = pd.read_parquet('challenge_candidates.parquet')
print(f'Loaded {len(df):,} candidates, {len(df.columns)} columns')
print('Columns:', df.columns.tolist()[:15], '...')

# Identify the candidate ID column
id_candidates = [c for c in df.columns if any(k in c.lower() for k in ('candidate_id','cand_id','candidateid'))]
ID_COL = id_candidates[0] if id_candidates else df.columns[0]
print(f'ID column: {ID_COL}')

# Text columns — same priority order as the pipeline schema detection
PREFERRED_TEXT_COLS = [
    'current_role', 'headline', 'summary', 'profile_text',
    'education', 'skills', 'experience_summary',
    'career_history_text', 'bio', 'about',
]
text_cols = [c for c in PREFERRED_TEXT_COLS if c in df.columns]
# Fallback: grab any remaining string columns (up to 12 total)
extra = [c for c in df.select_dtypes(include='object').columns
         if c not in text_cols and c != ID_COL]
text_cols = text_cols + extra[:max(0, 12 - len(text_cols))]
print(f'Text columns used ({len(text_cols)}): {text_cols}')

def build_doc(row):
    parts = []
    for col in text_cols:
        val = row.get(col)
        if val and str(val).strip() and str(val).lower() not in ('nan', 'none', ''):
            parts.append(str(val).strip())
    return ' '.join(parts)

print('Building candidate documents...')
candidate_ids = df[ID_COL].astype(str).tolist()
documents = [build_doc(row) for _, row in df.iterrows()]
print(f'Sample doc (first candidate): {documents[0][:200]}')

In [ ]:
# Cell 4 — Encode all candidates with BGE-large on GPU
import torch
from sentence_transformers import SentenceTransformer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

print('Loading BGE-large model...')
model = SentenceTransformer('BAAI/bge-large-en-v1.5', device=device)

print(f'Encoding {len(documents):,} candidates...')
embeddings = model.encode(
    documents,
    batch_size=256,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True,
)
embeddings = embeddings.astype('float32')
print(f'Embeddings shape: {embeddings.shape}')

In [ ]:
# Cell 5 — Build FAISS index and save both output files
import faiss
import pickle

dim = embeddings.shape[1]
print(f'Building FAISS flat IP index (dim={dim})...')

index = faiss.IndexFlatIP(dim)
index.add(embeddings)
print(f'Index contains {index.ntotal:,} vectors')

# Save FAISS index
faiss.write_index(index, 'dense_demo_index.faiss')
print('Saved: dense_demo_index.faiss')

# Save metadata (candidate IDs)
meta = {'candidate_ids': candidate_ids, 'index_type': 'flat'}
with open('dense_demo_index.meta.pkl', 'wb') as f:
    pickle.dump(meta, f)
print('Saved: dense_demo_index.meta.pkl')

import os
print(f'\nFile sizes:')
print(f'  dense_demo_index.faiss   : {os.path.getsize("dense_demo_index.faiss") / 1e6:.1f} MB')
print(f'  dense_demo_index.meta.pkl: {os.path.getsize("dense_demo_index.meta.pkl") / 1e6:.2f} MB')

In [ ]:
# Cell 6 — Download both files to your laptop
from google.colab import files
files.download('dense_demo_index.faiss')
files.download('dense_demo_index.meta.pkl')
print('Done! Copy both files to:  outputs/models/')

## After downloading

Copy both files to your local machine:
```
c:\Projects\India Runs\outputs\models\dense_demo_index.faiss
c:\Projects\India Runs\outputs\models\dense_demo_index.meta.pkl
```

Then generate the final submission:
```powershell
cd "c:\Projects\India Runs"
.venv\Scripts\activate
python scripts/generate_submission.py --recall-k 2000 --llm-rerank --validate
```